# Phase 1: TD(0) Value Learning — Random Policy (Partner Replication)

**Goal**: Learn $V_i(s)$ for each agent under random policy, validate against MC ground truth.
Batch TD(0) with partner's 2-mode transition system.

**Setup**: Run `sbatch run_mc.sh` first to generate MC ground truth, then Run All here.


In [7]:
# =============================================================================
# CONFIG (edit these, then Run All)
# =============================================================================

R_PICKER = -1
LAMBDA = 0               # 0 = TD(0)
MLP_LAYERS = [16, 16, 16, 16]

SEMI_MDP = True

LR = 0.01
LR_MIN = 1e-6
LR_DECAY_FRAC = 0.8

TRAIN_TICKS = 400_000     # sections (rounds). Each = all 4 agents × 2 modes.
EVAL_FREQ = 1000
CHECKPOINT_FREQ = 100_000
PRINT_FREQ = 10_000


MC_DEPTH = 1000
MC_TRAJ = 5
MC_ROLLOUTS = 200
NUM_MC_STATES = 1

MC_CACHE_DIR = "mc_cache_inline"

SEED = 42069
MC_DIR = "mc_cache"        # must match --output_dir in run_mc.sh
OUTPUT_PATH = "outputs"
TAG = f"td_lam{LAMBDA}_lr{LR}_rp{R_PICKER}"


In [8]:
# =============================================================================
# IMPORTS + DERIVED CONSTANTS
# =============================================================================

import os, sys, json, time, glob
import numpy as np
import pandas as pd
import torch

from config import (
    H, W, NUM_AGENTS, GAMMA, GAMMA_STEP, GAMMA_SEMI, K_NEAREST, INPUT_DIM,
    P_SPAWN, P_DESPAWN, L,
)
from helpers import (
    set_all_seeds, Reward, Orchard,
    encode_state, ValueNetwork, lr_factor,
    transition, evaluate_on_mc,
    get_memory_mb, get_weight_stats,
)

start_time = time.time()
R_OTHER = (1.0 - R_PICKER) / (NUM_AGENTS - 1)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {device}")
print(f"γ_round = {GAMMA}, γ_step = {GAMMA_STEP:.6f}")
print(f"input_dim = {INPUT_DIM}")
print(f"P_spawn = {P_SPAWN:.6f}, P_despawn = {P_DESPAWN:.6f}")
print(f"R_picker = {R_PICKER}, R_other = {R_OTHER:.4f}")
print(f"Each tick = 1 section = {NUM_AGENTS} agent-steps × 2 modes = {2*NUM_AGENTS} transitions/agent")


Device: cpu
γ_round = 0.99, γ_step = 0.998744
input_dim = 55
P_spawn = 0.012346, P_despawn = 0.031250
R_picker = -1, R_other = 0.6667
Each tick = 1 section = 4 agent-steps × 2 modes = 8 transitions/agent


In [9]:
# # =============================================================================
# # LOAD MC GROUND TRUTH  (generated by: sbatch run_mc.sh)
# # =============================================================================

# import glob

# mc_files = sorted(glob.glob("mc_cache/*.npz"))
# print(f"Found {len(mc_files)} MC state files")
# assert len(mc_files) > 0, "No eval states found"

# mc_states = []
# for f in mc_files:
#     with np.load(f, allow_pickle=True) as data:
#         mc_states.append(data["dict"].item())
        
# mc_mean_values = np.array([s["mc"] for s in mc_states])  # (num_states, NUM_AGENTS)

# avg_mag = np.mean(np.abs(mc_mean_values), axis=0)
# print(f"MC states loaded: {len(mc_states)}")
# for i in range(NUM_AGENTS):
#     vals = mc_mean_values[:, i]
#     print(f"  Agent {i}: mean={vals.mean():.4f}, std={vals.std():.4f}, "
#           f"avg|V|={avg_mag[i]:.4f}")


In [10]:
# =============================================================================
# GENERATE MC GROUND TRUTH (inline, cached)
# =============================================================================

import copy, random
from pathlib import Path
from helpers import mc_ground_truth

# Cache path encodes all MC-relevant parameters
mc_tag = (f"rp{R_PICKER}_g{GAMMA}_N{NUM_AGENTS}_d{MC_DEPTH}"
          f"_t{MC_TRAJ}_r{MC_ROLLOUTS}_seed{SEED}")
mc_cache_path = Path(MC_CACHE_DIR) / mc_tag
mc_cache_path.mkdir(parents=True, exist_ok=True)

mc_states = []
num_cached = 0

for idx in range(NUM_MC_STATES):
    fpath = mc_cache_path / f"state_{idx}.npz"
    if fpath.exists():
        with np.load(fpath, allow_pickle=True) as data:
            st_mc = data["state_dict"].item()
        mc_states.append(st_mc)
        num_cached += 1
        print(f"State {idx}: loaded from cache")
        continue

    set_all_seeds(SEED + idx)
    reward_mod_mc = Reward(R_PICKER, NUM_AGENTS)
    orch_mc = Orchard(W, W, NUM_AGENTS, reward_mod_mc, p_apple=0.2, d_apple=P_DESPAWN)
    orch_mc.p_apple = P_SPAWN
    orch_mc.set_positions()
    st_mc = dict(orch_mc.get_state())
    st_mc["actor_id"] = random.randint(0, NUM_AGENTS - 1)
    st_mc["mode"] = 0

    mc_standard = mc_ground_truth(
        SEED + idx, MC_DEPTH, orch_mc, st_mc, GAMMA_STEP,
        MC_TRAJ, MC_ROLLOUTS, semi_mdp=False,
    )
    mc_semi = mc_ground_truth(
        SEED + idx, MC_DEPTH, orch_mc, st_mc, GAMMA_SEMI,
        MC_TRAJ, MC_ROLLOUTS, semi_mdp=True,
    )

    st_mc["mc_standard"] = mc_standard
    st_mc["mc_semi"] = mc_semi

    print(f"State {idx}: apples={st_mc['apples'].sum()}, actor={st_mc['actor_id']}")
    for i in range(NUM_AGENTS):
        print(f"  Agent {i}: standard={mc_standard[i]:.6f}, semi={mc_semi[i]:.6f}, Δ={mc_semi[i]-mc_standard[i]:.6f}")

    np.savez_compressed(fpath, state_dict=np.array(st_mc, dtype=object))
    mc_states.append(st_mc)

# Select active MC values based on SEMI_MDP flag
for st in mc_states:
    st["mc"] = st["mc_semi"] if SEMI_MDP else st["mc_standard"]

mc_mean_values = np.array([s["mc"] for s in mc_states])
print(f"\n{NUM_MC_STATES} states ({num_cached} cached, {NUM_MC_STATES - num_cached} generated)")
print(f"Using {'semi-MDP' if SEMI_MDP else 'standard MDP'} ground truth")
print(f"Cache: {mc_cache_path}")

State 0: loaded from cache

1 states (1 cached, 0 generated)
Using semi-MDP ground truth
Cache: mc_cache_inline/rp-1_g0.99_N4_d1000_t5_r200_seed42069


In [11]:
# =============================================================================
# CREATE MODELS (one per agent)
# =============================================================================

set_all_seeds(SEED)

models = []
optimizers = []
for i in range(NUM_AGENTS):
    m = ValueNetwork(INPUT_DIM, MLP_LAYERS).to(device)
    models.append(m)
    optimizers.append(torch.optim.SGD(m.parameters(), lr=LR))

total_params = sum(p.numel() for p in models[0].parameters())
print(f"Model: {MLP_LAYERS}, {total_params} params per agent")
print(models[0])

# LR scheduler (linear decay over 80%, then hold)
decay_steps = int(TRAIN_TICKS * LR_DECAY_FRAC)
min_factor = (LR_MIN / LR) if LR > 0 else 0.0
schedulers = [
    torch.optim.lr_scheduler.LambdaLR(
        opt, lr_lambda=lambda s: lr_factor(s, decay_steps, min_factor)
    )
    for opt in optimizers
]


Model: [16, 16, 16, 16], 1729 params per agent
ValueNetwork(
  (net): Sequential(
    (0): Linear(in_features=55, out_features=16, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=16, out_features=16, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=16, out_features=16, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=16, out_features=16, bias=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [12]:
# =============================================================================
# TRAINING LOOP
# =============================================================================

os.makedirs(OUTPUT_PATH, exist_ok=True)
csv_path = os.path.join(OUTPUT_PATH, f"phase1_{TAG}.csv")
meta_path = csv_path.replace('.csv', '_metadata.json')
ckpt_dir = os.path.join(OUTPUT_PATH, f"checkpoints_{TAG}")
os.makedirs(ckpt_dir, exist_ok=True)

# Metadata
metadata = {
    "phase": 1, "R_PICKER": R_PICKER, "R_OTHER": R_OTHER,
    "LAMBDA": LAMBDA, "MLP_LAYERS": MLP_LAYERS,
    "LR": LR, "LR_MIN": LR_MIN, "LR_DECAY_FRAC": LR_DECAY_FRAC,
    "TRAIN_TICKS": TRAIN_TICKS, "EVAL_FREQ": EVAL_FREQ,
    "SEED": SEED, "H": H, "W": W, "NUM_AGENTS": NUM_AGENTS,
    "GAMMA": GAMMA, "GAMMA_STEP": GAMMA_STEP,
    "K_NEAREST": K_NEAREST, "INPUT_DIM": INPUT_DIM, "L": L,
    "P_SPAWN": P_SPAWN, "P_DESPAWN": P_DESPAWN,
    "num_mc_states": len(mc_states),
    "avg_magnitude_true_value": {
        f"agent_{i}": float(np.mean(np.abs(mc_mean_values[:, i])))
        for i in range(NUM_AGENTS)
    },
}
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=4)

# Initialize environment
set_all_seeds(SEED)
reward_mod = Reward(R_PICKER, NUM_AGENTS)
env = Orchard(W, W, NUM_AGENTS, reward_mod, P_SPAWN, P_DESPAWN)
env.set_positions()

curr_state = None
actor_idx = None
first_write = True
prev_weights = {i: [] for i in range(NUM_AGENTS)}
total_grad_steps = 0
t0 = time.time()
gamma_m0 = GAMMA_SEMI if SEMI_MDP else GAMMA_STEP   # mode 0→1
gamma_m1 = 1.0 if SEMI_MDP else GAMMA_STEP           # mode 1→0

# Per-agent batch buffers
batch_states = {i: [] for i in range(NUM_AGENTS)}
batch_new_states = {i: [] for i in range(NUM_AGENTS)}
batch_rewards = {i: [] for i in range(NUM_AGENTS)}
batch_gammas = {i: [] for i in range(NUM_AGENTS)}

# Initial eval
metrics = evaluate_on_mc(models, mc_states, device)
print(f"Tick 0 (init) | pct_err={metrics['pct_err_total']:.2f}%")

print(f"Training: {TRAIN_TICKS} ticks, λ={LAMBDA}")
print(f"Logging to: {csv_path}")

for tick in range(TRAIN_TICKS):
    # --- Collect experiences for this section ---
    for step in range(-1, NUM_AGENTS):
        final_state, semi_state, res, actor_idx = transition(
            step, curr_state, env, actor_idx
        )
        if step != -1:
            # ALL agents observe EVERY agent's step: 2 experiences each
            for i in range(NUM_AGENTS):
                enc_old = encode_state(curr_state, i)
                enc_semi = encode_state(semi_state, i)
                enc_final = encode_state(final_state, i)

                # mode 0 → mode 1: reward = 0
                batch_states[i].append(enc_old)
                batch_new_states[i].append(enc_semi)
                batch_rewards[i].append(0.0)
                batch_gammas[i].append(gamma_m0)

                # mode 1 → mode 0: reward = agent i's share
                batch_states[i].append(enc_semi)
                batch_new_states[i].append(enc_final)
                batch_rewards[i].append(res.reward_vector[i])
                batch_gammas[i].append(gamma_m1)

        curr_state = final_state

    # --- Batch TD(0) update ---
    for i in range(NUM_AGENTS):
        B = len(batch_states[i])
        states_t = torch.as_tensor(
            np.stack(batch_states[i]), device=device
        ).view(B, -1)
        nxt_t = torch.as_tensor(
            np.stack(batch_new_states[i]), device=device
        ).view(B, -1)
        rewards_t = torch.as_tensor(
            np.array(batch_rewards[i], dtype=np.float64), device=device
        )

        pred = models[i](states_t).squeeze(-1)
        with torch.no_grad():
            v_next = models[i](nxt_t).squeeze(-1)
            gammas_t = torch.as_tensor(
            np.array(batch_gammas[i], dtype=np.float64), device=device
            )           
            target = rewards_t + gammas_t * v_next

        loss = torch.nn.functional.mse_loss(pred, target)
        optimizers[i].zero_grad()
        loss.backward()
        optimizers[i].step()
        schedulers[i].step()

        batch_states[i].clear()
        batch_new_states[i].clear()
        batch_rewards[i].clear()
        batch_gammas[i].clear()

    total_grad_steps += 1

    # --- Evaluate ---
    if (tick + 1) % EVAL_FREQ == 0:
        metrics = evaluate_on_mc(models, mc_states, device)
        current_lr = optimizers[0].param_groups[0]['lr']
        row = {'tick': tick + 1, 'total_grad_steps': total_grad_steps,
               'current_lr': current_lr, 'memory_mb': get_memory_mb()}
        row.update(metrics)

        for i in range(NUM_AGENTS):
            ws, prev_weights[i] = get_weight_stats(models[i], prev_weights[i])
            for k, v in ws.items():
                row[f'a{i}_{k}'] = v

        pd.DataFrame([row]).to_csv(
            csv_path, mode='w' if first_write else 'a',
            header=first_write, index=False,
        )
        first_write = False

    # --- Print ---
    if (tick + 1) % PRINT_FREQ == 0:
        elapsed = time.time() - t0
        metrics = evaluate_on_mc(models, mc_states, device)
        current_lr = optimizers[0].param_groups[0]['lr']
        print(f"Tick {tick+1:>8d} | grad_steps={total_grad_steps} | "
              f"pct_err={metrics['pct_err_total']:.2f}% | "
              f"mae={metrics['mae_total']:.4f} | "
              f"bias={metrics['bias_total']:.4f} | "
              f"lr={current_lr:.2e} | {elapsed:.0f}s")

    # --- Checkpoint ---
    if (tick + 1) % CHECKPOINT_FREQ == 0:
        for i in range(NUM_AGENTS):
            torch.save(models[i].state_dict(),
                       os.path.join(ckpt_dir, f"agent{i}_tick{tick+1}.pt"))

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s ({total_grad_steps} grad steps)")


Tick 0 (init) | pct_err=103.27%
Training: 400000 ticks, λ=0
Logging to: outputs/phase1_td_lam0_lr0.01_rp-1.csv
Tick    10000 | grad_steps=10000 | pct_err=18.49% | mae=1.9475 | bias=-0.5379 | lr=9.69e-03 | 555s
Tick    20000 | grad_steps=20000 | pct_err=17.69% | mae=2.0357 | bias=-1.7639 | lr=9.38e-03 | 1095s


KeyboardInterrupt: 

In [ ]:
# =============================================================================
# FINAL EVALUATION
# =============================================================================

end_time = time.time()
time_taken = end_time - start_time
time_min = time_taken / 60
print(f"Total time taken: {time_taken:.1f}s ({time_min:.1f}min)")
metadata['total_time_seconds'] = time_taken
metadata['total_time_minutes'] = time_min

with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)

metrics = evaluate_on_mc(models, mc_states, device)

for i in range(NUM_AGENTS):
    print(f"\nAgent {i}:")
    print(f"  MAE:       {metrics[f'a{i}_mae']:.6f}")
    print(f"  Bias:      {metrics[f'a{i}_bias']:.6f}")
    print(f"  % Error:   {metrics[f'a{i}_pct_error']:.2f}%")
    print(f"  Mean Pred: {metrics[f'a{i}_mean_pred']:.6f}")
    print(f"  Mean True: {metrics[f'a{i}_mean_true']:.6f}")

print(f"\nOverall MAE:     {metrics['mae_total']:.6f}")
print(f"Overall % Error: {metrics['pct_err_total']:.2f}%")

if metrics['pct_err_total'] < 10.0:
    print("*** PASS: <10% error ***")
else:
    print("*** FAIL: >=10% error ***")

# Save final models
for i in range(NUM_AGENTS):
    torch.save(models[i].state_dict(),
               os.path.join(OUTPUT_PATH, f"final_agent{i}_{TAG}.pt"))
print(f"\nFinal models saved to {OUTPUT_PATH}/")


In [ ]:
# =============================================================================
# PLOT
# =============================================================================

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} eval rows from {csv_path}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df['tick'], df['pct_err_total'], 'o-', linewidth=2, markersize=3, label='Overall')

for i in range(NUM_AGENTS):
    col = f'a{i}_pct_error'
    if col in df.columns:
        ax.plot(df['tick'], df[col], '.-', linewidth=1, alpha=0.6, label=f'Agent {i}')

ax.set_xlabel("Training tick (section)")
ax.set_ylabel("% Error (MAE / |true|)")
ax.set_title(f"TD(0) Replication | γ={GAMMA}, α={LR}, picker_r={R_PICKER}")
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
ax.legend(ncol=3, fontsize=9)
fig.tight_layout()

plot_path = os.path.join(OUTPUT_PATH, f"phase1_{TAG}.png")
fig.savefig(plot_path, dpi=250, bbox_inches="tight")
print(f"Plot saved to {plot_path}")
plt.show()
